First of all, we init the SPARQLWrapper service with the SPARQL endpoint

In [ ]:
!pip install SPARQLWrapper

In [11]:
import csv
from SPARQLWrapper import SPARQLWrapper

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

Then we define our CONSTRUCT query to extract the metadata

In [2]:
sparql.setQuery("""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>

CONSTRUCT {
  wd:Q535 wdt:P31 ?type .
  wd:Q535 wdt:P18 ?image .
  wd:Q535 wdt:P27 ?citizenship .
  wd:Q535 wdt:P569 ?dateBirth .
  ?citizenship rdfs:label ?citizenshipLabel .
  wd:Q535 wdt:P19 ?placeBirth .
  ?placeBirth rdfs:label ?placeBirthLabel .
  wd:Q535 wdt:P19 ?placeDeath .
  ?placeDeath rdfs:label ?placeDeathLabel .
  wd:Q535 wdt:P119 ?burial .
  ?burial rdfs:label ?burialLabel .
  wd:Q535 wdt:P1442 ?graveImage .
  wd:Q535 wdt:P103 ?language.
  ?language rdfs:label ?languageLabel .
  wd:Q535 wdt:P106 ?occupation.
  ?occupation rdfs:label ?occupationLabel .
  wd:Q535 wdt:P135 ?movement.
  ?movement rdfs:label ?movementLabel .
  wd:Q535 wdt:P800 ?work.
  ?work rdfs:label ?workLabel .
}
WHERE {
  wd:Q535 wdt:P31 ?type .
  wd:Q535 wdt:P18 ?image .
  wd:Q535 wdt:P27 ?citizenship .
  wd:Q535 wdt:P569 ?dateBirth .     
  wd:Q535 wdt:P19 ?placeBirth .
  ?citizenship rdfs:label ?citizenshipLabel  FILTER (lang(?citizenshipLabel) = 'en') .
  ?placeBirth rdfs:label ?placeBirthLabel  FILTER (lang(?placeBirthLabel) = 'en') .
  wd:Q535 wdt:P570 ?dateDeath .     
  wd:Q535 wdt:P20 ?placeDeath .
  ?placeDeath rdfs:label ?placeDeathLabel  FILTER (lang(?placeDeathLabel) = 'en') .
  wd:Q535 wdt:P509 ?causeDeath .
  ?causeDeath rdfs:label ?causeDeathLabel  FILTER (lang(?causeDeathLabel) = 'en') .
  wd:Q535 wdt:P119 ?burial .
  ?burial rdfs:label ?burialLabel  FILTER (lang(?burialLabel) = 'en') .
  wd:Q535 wdt:P1442 ?graveImage .
  wd:Q535 wdt:P103 ?language .
  ?language rdfs:label ?languageLabel  FILTER (lang(?languageLabel) = 'en') .
  wd:Q535 wdt:P106 ?occupation .
  ?occupation rdfs:label ?occupationLabel  FILTER (lang(?occupationLabel) = 'en') .
  wd:Q535 wdt:P135 ?movement .
  ?movement rdfs:label ?movementLabel  FILTER (lang(?movementLabel) = 'fr') .
  wd:Q535 wdt:P800 ?work .
  ?work rdfs:label ?workLabel  FILTER (lang(?workLabel) = 'fr') .
}
"""
)

Finally, we serialise the result

In [3]:
results = sparql.queryAndConvert()
print(results.serialize())

@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix wd: <http://www.wikidata.org/entity/> .
@prefix wdt: <http://www.wikidata.org/prop/direct/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

wd:Q535 wdt:P103 wd:Q150 ;
    wdt:P106 wd:Q11774202,
        wd:Q15296811,
        wd:Q1792450,
        wd:Q1925963,
        wd:Q214917,
        wd:Q3579035,
        wd:Q36180,
        wd:Q482980,
        wd:Q49757,
        wd:Q6051619,
        wd:Q644687,
        wd:Q6625963,
        wd:Q8178443,
        wd:Q82955 ;
    wdt:P119 wd:Q188856 ;
    wdt:P135 wd:Q27301872 ;
    wdt:P1442 <http://commons.wikimedia.org/wiki/Special:FilePath/Pante%C3%B3n%2C%20Par%C3%ADs%2C%20Francia%2C%202022-10-29%2C%20DD%2049-51%20HDR.jpg> ;
    wdt:P18 <http://commons.wikimedia.org/wiki/Special:FilePath/Victor%20Hugo%20by%20%C3%89tienne%20Carjat%201876%20-%20full.jpg> ;
    wdt:P19 wd:Q37776,
        wd:Q90 ;
    wdt:P27 wd:Q142 ;
    wdt:P31 wd:Q5 ;
    wdt:P569 "1802-02-26T00:00:00+00:00"^^xsd:dat

In [4]:
with open("output/victor-hugo.ttl", "w") as text_file:
    text_file.write(results.serialize())

#### We can also run a federated SPARQL query to retrieve integrated data from external repositories such as the National Library of France.

In [13]:
query = """
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdarelationships: <http://rdvocab.info/RDARelationshipsWEMI/>
PREFIX rdagroup1elements: <http://rdvocab.info/Elements/>

SELECT DISTINCT ?author ?expression ?title ?edition ?placeOfPublication ?yearOfPublication ?langCode WHERE {
wd:Q535 wdt:P268 ?id
BIND(uri(concat(concat("http://data.bnf.fr/ark:/12148/cb", ?id),"#about")) as ?author)
SERVICE <http://data.bnf.fr/sparql> {
  ?expression <http://id.loc.gov/vocabulary/relators/aut> ?author .
  OPTIONAL {?expression dcterms:language ?langCode .}
  OPTIONAL {?manifestation dcterms:publisher ?edition .}
  ?manifestation rdarelationships:expressionManifested ?expression .
  ?manifestation dcterms:title ?title .
  ?manifestation dcterms:date ?yearOfPublication .
  OPTIONAL{ ?manifestation rdagroup1elements:placeOfPublication ?placeOfPublication .}
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }

}
LIMIT 5000
"""

In [14]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql.setReturnFormat(JSON)

sparql.setQuery(query)

In [15]:
try:
    ret = sparql.queryAndConvert()

    for r in ret["results"]["bindings"]:
        print(r)
except Exception as e:
    print(e)

{'author': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb11907966z#about'}, 'yearOfPublication': {'type': 'literal', 'value': '1990'}, 'expression': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb35087521r#Expression'}, 'placeOfPublication': {'type': 'literal', 'value': 'Paris'}, 'langCode': {'type': 'uri', 'value': 'http://id.loc.gov/vocabulary/iso639-2/fre'}, 'edition': {'type': 'literal', 'value': 'Paris : Hachette , 1990'}, 'title': {'type': 'literal', 'value': 'Fantine'}}
{'author': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb11907966z#about'}, 'yearOfPublication': {'type': 'literal', 'value': '1992'}, 'expression': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb35499435j#Expression'}, 'placeOfPublication': {'type': 'literal', 'value': 'Paris'}, 'langCode': {'type': 'uri', 'value': 'http://id.loc.gov/vocabulary/iso639-2/fre'}, 'edition': {'type': 'literal', 'value': 'Paris : Hachette jeunesse , 1992'}, 'title': {'type': 'literal', 'v

Now we can create a CSV file with the content

In [19]:
output_file="output/victor-hugo-works.csv"
with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["author", "expression", "title", "edition", "placeOfPublication", "yearOfPublication", "langCode"])

    for item in ret["results"]["bindings"]:
        author = item.get("author", {}).get("value", "")
        expression = item.get("expression", {}).get("value", "")
        title = item.get("title", {}).get("value", "")
        edition = item.get("edition", {}).get("value", "")
        placeOfPublication = item.get("placeOfPublication", {}).get("value", "")
        yearOfPublication = item.get("yearOfPublication", {}).get("value", "")
        langCode = item.get("langCode", {}).get("value", "")

        writer.writerow([author, expression, title, edition, placeOfPublication, yearOfPublication, langCode])

#### Now we can analyse the content retrieved

In [21]:
import pandas as pd

# Load the CSV
df = pd.read_csv(output_file)

# Show first rows
print("First rows:")
print(df.head())

# Convert publication date to datetime
df["yearOfPublication"] = pd.to_datetime(df["yearOfPublication"], errors="coerce")

# Extract year
df["Year"] = df["yearOfPublication"].dt.year

# Basic info
print("\nData info:")
print(df.info())

# Count works per year
works_per_year = df["Year"].value_counts().sort_index()

print("\nWorks per year:")
print(works_per_year)

# Find earliest and latest works
earliest = df.loc[df["Year"].idxmin()]
latest = df.loc[df["Year"].idxmax()]

print("\nEarliest work:")
print(earliest)

print("\nLatest work:")
print(latest)

# Save summary to new CSV
works_per_year.to_csv("output/victor-hugo-works_per_year.csv")

First rows:
                                            author  \
0  http://data.bnf.fr/ark:/12148/cb11907966z#about   
1  http://data.bnf.fr/ark:/12148/cb11907966z#about   
2  http://data.bnf.fr/ark:/12148/cb11907966z#about   
3  http://data.bnf.fr/ark:/12148/cb11907966z#about   
4  http://data.bnf.fr/ark:/12148/cb11907966z#about   

                                          expression  \
0  http://data.bnf.fr/ark:/12148/cb35087521r#Expr...   
1  http://data.bnf.fr/ark:/12148/cb35499435j#Expr...   
2  http://data.bnf.fr/ark:/12148/cb485339933#Expr...   
3  http://data.bnf.fr/ark:/12148/cb42953819p#Expr...   
4  http://data.bnf.fr/ark:/12148/cb47573737r#Expr...   

                                               title  \
0                                            Fantine   
1                                           Gavroche   
2  Bug-Jargal : une épopée passionnante de libert...   
3  The Children of the poor.. Poem by Victor Hugo...   
4                                     Les misé

#### Now get all the distinct places of publication

In [23]:
unique_values = df["placeOfPublication"].unique()
print(unique_values)

['Paris' 'Forbach' 'New York' 'Lyon' 'Vanves' '[Paris]' '[Paris?]'
 'Les Auxons' 'Chaillot' 'Paris. - [France]' 'Strasbourg' '[S.l.]' 'S.l.'
 'Leipzig' 'Bratislava' 'Tampere' '[Paris]. - [Paris'
 'Londres. - Bruxelles' 'Saint-Helier' 'Paris. - Barcelone. - Lausanne'
 'Paris. - [Paris]' '[Paris]. - [Montrouge]' 'Paris [et al]'
 'Grenoble. - Grenoble' 'France' 'Apt' 'Genève. - Évreux' 'Les Hermites'
 'Amiens' 'Arles' 'Bruxelles' 'Montrouge' '[Neuilly-sur-Seine]' 'London'
 '[Paris' 'Genève. - Évreux. - [1978]' '[Pau]' 'K. Polis' 'A Paris'
 'Villemer' '[Lieu de publication inconnu]' 'Paris. - Paris' '(S. l.'
 'Sampzon' 'Chevron (Belgique). - [Pantin]' 'Darnétal' 'Avignon'
 'Vénissieux' '(Paris,)' 'Neuilly-sur-Seine. - Paris'
 'Neuilly-sur-Seine. - [Paris]' '[Saint-Cloud]. - [Paris]'
 'Paris. - Livraphone. - [France]' 'Saint-Chéron' 'Arles. - Arles'
 'Genève. - New-York' 'Paris. - [S.l.]'
 'Paris, Hachette. - (Coulommiers, impr. de Brodard et Taupin)'
 'Genève. - [La Seyne-sur-Mer]' "[Saint

#### Now get all the languages

In [24]:
unique_values = df["langCode"].unique()
print(unique_values)

['http://id.loc.gov/vocabulary/iso639-2/fre'
 'http://id.loc.gov/vocabulary/iso639-2/eng'
 'http://id.loc.gov/vocabulary/iso639-2/zxx' nan
 'http://id.loc.gov/vocabulary/iso639-2/slo'
 'http://id.loc.gov/vocabulary/iso639-2/arm'
 'http://id.loc.gov/vocabulary/iso639-2/vie'
 'http://id.loc.gov/vocabulary/iso639-2/ger'
 'http://id.loc.gov/vocabulary/iso639-2/spa'
 'http://id.loc.gov/vocabulary/iso639-2/lat'
 'http://id.loc.gov/vocabulary/iso639-2/gre'
 'http://id.loc.gov/vocabulary/iso639-2/ita'
 'http://id.loc.gov/vocabulary/iso639-2/rum'
 'http://id.loc.gov/vocabulary/iso639-2/por'
 'http://id.loc.gov/vocabulary/iso639-2/dut'
 'http://id.loc.gov/vocabulary/iso639-2/pol'
 'http://id.loc.gov/vocabulary/iso639-2/tur'
 'http://id.loc.gov/vocabulary/iso639-2/ukr'
 'http://id.loc.gov/vocabulary/iso639-2/cze'
 'http://id.loc.gov/vocabulary/iso639-2/hun'
 'http://id.loc.gov/vocabulary/iso639-2/rus'
 'http://id.loc.gov/vocabulary/iso639-2/urd'
 'http://id.loc.gov/vocabulary/iso639-2/alb'
 'http

In [26]:
# Count works per language
works_per_language = df["langCode"].value_counts().sort_index()

print("\nWorks per language:")
print(works_per_language)


Works per language:
langCode
http://id.loc.gov/vocabulary/iso639-2/alb       1
http://id.loc.gov/vocabulary/iso639-2/ara       1
http://id.loc.gov/vocabulary/iso639-2/arm       2
http://id.loc.gov/vocabulary/iso639-2/chi       1
http://id.loc.gov/vocabulary/iso639-2/cze      15
http://id.loc.gov/vocabulary/iso639-2/dut       2
http://id.loc.gov/vocabulary/iso639-2/eng      63
http://id.loc.gov/vocabulary/iso639-2/fre    3767
http://id.loc.gov/vocabulary/iso639-2/ger      24
http://id.loc.gov/vocabulary/iso639-2/gre       1
http://id.loc.gov/vocabulary/iso639-2/hun       1
http://id.loc.gov/vocabulary/iso639-2/ita      10
http://id.loc.gov/vocabulary/iso639-2/lat       3
http://id.loc.gov/vocabulary/iso639-2/pol       2
http://id.loc.gov/vocabulary/iso639-2/por       4
http://id.loc.gov/vocabulary/iso639-2/rum       5
http://id.loc.gov/vocabulary/iso639-2/rus       7
http://id.loc.gov/vocabulary/iso639-2/slo       1
http://id.loc.gov/vocabulary/iso639-2/spa      35
http://id.loc.gov/vo

#### References
- Candela, G. (2023). Towards a semantic approach in GLAM Labs: The case of the Data Foundry at the National Library of Scotland. Journal of Information Science, 52(1), 3–21. https://doi.org/10.1177/01655515231174386
- Dişli, M., Osti, G., Candela, G., & Zijdeman, R. (2025). From Linked Open Data to Collections as Data: A Reproducible Framework Using Federated Queries. Information Technology and Libraries, 44(4). https://doi.org/10.5860/ital.v44i4.17432